# 17. Frozen MERT 초기 모델 (v2)

MERT 표현과 LR로 REAL/FAKE를 구분한 과거 실험이다. 이 버전은 Transformers 호환성과 Apple Silicon 실행 문제를 해결해 MERT embedding을 만들었다. 최신 17B·23번 결과와 구분해 읽는다.

## 0. 최초 1회 설치

이 셀을 실행한 뒤 **VSCode에서 같은 Python 3.13.9 커널을 Restart**하고 1번부터 실행하세요.

In [1]:
# 과거 MERT 실험의 고정된 패키지 버전을 현재 커널에 설치한다.
%pip install -q "transformers==4.47.1" "tokenizers==0.21.0" "huggingface_hub==0.27.1" "safetensors>=0.4.5"
print("설치 완료. 이제 Restart Kernel 후 1번 셀부터 실행하세요.")

Note: you may need to restart the kernel to use updated packages.
설치 완료. 이제 Restart Kernel 후 1번 셀부터 실행하세요.


## 1. 환경 / 경로

In [2]:
import os

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

from pathlib import Path
import random, time, gc
import numpy as np
import pandas as pd
import librosa
import torch
import joblib
import transformers

from transformers import AutoConfig, AutoModel, Wav2Vec2FeatureExtractor
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_curve,
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
)

PROJECT_ROOT = Path("/Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project")
SEGMENT_PATH = PROJECT_ROOT / "data/metadata/segment_manifest_10s.csv"

MERT_DIR = PROJECT_ROOT / "data/processed/mert"
RESULT_DIR = PROJECT_ROOT / "results/mert"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints/mert"
for p in (MERT_DIR, RESULT_DIR, CHECKPOINT_DIR):
    p.mkdir(parents=True, exist_ok=True)

EMBED_PATH = MERT_DIR / "mert95m_v2_layers_float16.npy"
DONE_PATH = MERT_DIR / "mert95m_v2_done.npy"
INDEX_PATH = MERT_DIR / "mert95m_v2_index.csv"

MODEL_NAME = "m-a-p/MERT-v1-95M"
MERT_REVISION = "12af15fef9d0ac838c3f475bfbbf26d2060dd4f5"

SR = 24000
SEGMENT_SEC = 10.0
TARGET_SAMPLES = int(SR * SEGMENT_SEC)
N_REP_LAYERS = 13
HIDDEN_SIZE = 768
RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

print("PyTorch      :", torch.__version__)
print("Transformers :", transformers.__version__)
print(
    "MPS available:",
    hasattr(torch.backends, "mps") and torch.backends.mps.is_available(),
)

if transformers.__version__ != "4.47.1":
    raise RuntimeError(
        f"현재 Transformers={transformers.__version__}. "
        "0번 셀 실행 후 Restart Kernel이 필요합니다."
    )

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

PyTorch      : 2.14.0
Transformers : 4.47.1
MPS available: True


## 2. Segment Manifest / Audio QC

In [3]:
segments = pd.read_csv(SEGMENT_PATH).reset_index(drop=True)

required = [
    "segment_id",
    "track_sample_id",
    "original_audio",
    "label",
    "label_id",
    "genre",
    "generator",
    "split",
    "start_sec",
    "audio_path",
]
missing = [c for c in required if c not in segments.columns]
if missing:
    raise ValueError("필수 컬럼 없음: " + ", ".join(missing))

print("Rows:", len(segments))
# 같은 원곡의 표본이 여러 분할에 섞이지 않도록 저장된 split을 그대로 사용한다.
print(segments["split"].value_counts())

# ID와 행 수가 틀리면 특징·라벨 대응이 어긋나므로 여기서 확인한다.
assert len(segments) == 10077
assert segments["split"].isin(["train", "val", "test"]).all()
assert segments["label_id"].isin([0, 1]).all()


def load_segment_waveform(row):
    full_path = PROJECT_ROOT / str(row["audio_path"])
    if not full_path.exists():
        raise FileNotFoundError(full_path)

    y, _ = librosa.load(
        full_path,
        sr=SR,
        mono=True,
        offset=float(row["start_sec"]),
        duration=SEGMENT_SEC,
    )
    y = np.asarray(y, dtype=np.float32)

    if len(y) < TARGET_SAMPLES:
        y = np.pad(y, (0, TARGET_SAMPLES - len(y)))
    else:
        y = y[:TARGET_SAMPLES]

    if len(y) != TARGET_SAMPLES:
        raise RuntimeError(f"waveform length mismatch: {len(y)}")
    return y


sample_y = load_segment_waveform(segments.iloc[0])

print("Sample waveform:", sample_y.shape, sample_y.dtype)
print("finite:", bool(np.isfinite(sample_y).all()))

assert sample_y.shape == (240000,)
assert np.isfinite(sample_y).all()
print("Audio Loader QC PASS: True")

Rows: 10077
split
train    6967
test     1572
val      1538
Name: count, dtype: int64
Sample waveform: (240000,) float32
finite: True
Audio Loader QC PASS: True


## 3. MERT 안전 로드

In [4]:
print("Loading:", MODEL_NAME)
print("Revision:", MERT_REVISION)

config = AutoConfig.from_pretrained(
    MODEL_NAME,
    revision=MERT_REVISION,
    trust_remote_code=True,
)
config.conv_pos_batch_norm = False

processor = Wav2Vec2FeatureExtractor.from_pretrained(
    MODEL_NAME,
    revision=MERT_REVISION,
    trust_remote_code=True,
)

model = AutoModel.from_pretrained(
    MODEL_NAME,
    revision=MERT_REVISION,
    config=config,
    trust_remote_code=True,
)

model = model.to("cpu")
model.eval()
for p in model.parameters():
    p.requires_grad = False

print("Processor sample rate:", processor.sampling_rate)
print("Config sample rate   :", getattr(config, "sample_rate", None))
print("Hidden size          :", getattr(config, "hidden_size", None))
print("Hidden layers        :", getattr(config, "num_hidden_layers", None))
print("Model params         :", sum(p.numel() for p in model.parameters()))
print("Frozen               :", all(not p.requires_grad for p in model.parameters()))

# ID와 행 수가 틀리면 특징·라벨 대응이 어긋나므로 여기서 확인한다.
assert processor.sampling_rate == 24000
assert int(config.sample_rate) == 24000
assert int(config.hidden_size) == 768
assert int(config.num_hidden_layers) == 12

print("MERT Load QC PASS: True")

Loading: m-a-p/MERT-v1-95M
Revision: 12af15fef9d0ac838c3f475bfbbf26d2060dd4f5
Processor sample rate: 24000
Config sample rate   : 24000
Hidden size          : 768
Hidden layers        : 12
Model params         : 94371712
Frozen               : True
MERT Load QC PASS: True


## 4. 단일 샘플 Forward + MPS 자동 fallback

In [5]:
def extract_mert_embeddings(y, device):
    inputs = processor(
        y,
        sampling_rate=SR,
        return_tensors="pt",
        padding=False,
    )

    input_values = inputs["input_values"].to(
        device=device,
        dtype=torch.float32,
    )

    attention_mask = inputs.get("attention_mask")
    if attention_mask is not None:
        attention_mask = attention_mask.to(device)

    with torch.inference_mode():
        outputs = model(
            input_values=input_values,
            attention_mask=attention_mask,
            output_hidden_states=True,
            return_dict=True,
        )

    hidden_states = outputs.hidden_states
    if hidden_states is None:
        raise RuntimeError("hidden_states가 반환되지 않았습니다.")
    if len(hidden_states) != 13:
        raise RuntimeError(f"hidden state count={len(hidden_states)}")

    pooled = torch.stack(
        # 시간 frame의 표현을 평균해 Segment 하나의 고정 길이 벡터로 만든다.
        [h.float().mean(dim=1).squeeze(0) for h in hidden_states],
        dim=0,
    )
    return pooled.detach().cpu().numpy().astype(np.float32)


mps_ok = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
candidate = torch.device("mps" if mps_ok else "cpu")

try:
    model = model.to(candidate)
    sample_emb = extract_mert_embeddings(sample_y, candidate)
    MERT_DEVICE = candidate
    print("Forward success on:", MERT_DEVICE)
except Exception as e:
    if candidate.type != "mps":
        raise
    print("MPS forward failed:", type(e).__name__, str(e)[:500])
    print("CPU로 자동 fallback 합니다.")
    try:
        model = model.to("cpu")
    except Exception:
        pass
    try:
        torch.mps.empty_cache()
    except Exception:
        pass
    gc.collect()
    MERT_DEVICE = torch.device("cpu")
    model = model.to(MERT_DEVICE)
    sample_emb = extract_mert_embeddings(sample_y, MERT_DEVICE)

print("Waveform shape :", sample_y.shape)
print("MERT embedding :", sample_emb.shape)
print("dtype          :", sample_emb.dtype)
print("finite         :", bool(np.isfinite(sample_emb).all()))
print("Final device   :", MERT_DEVICE)

# ID와 행 수가 틀리면 특징·라벨 대응이 어긋나므로 여기서 확인한다.
assert sample_emb.shape == (13, 768)
assert np.isfinite(sample_emb).all()
print("Single-sample MERT QC PASS: True")

Forward success on: mps
Waveform shape : (240000,)
MERT embedding : (13, 768)
dtype          : float32
finite         : True
Final device   : mps
Single-sample MERT QC PASS: True


## 5. 전체 10,077개 embedding 추출

**4번까지 정상일 때만 실행하세요.**  
중간 중단 후 재실행하면 `done.npy`를 이용해 이어서 진행합니다.

In [6]:
shape = (len(segments), 13, 768)

if EMBED_PATH.exists():
    # 배열 캐시를 읽어 저장된 특징과 ID의 순서를 확인한다.
    chk = np.load(EMBED_PATH, mmap_mode="r")
    if chk.shape != shape:
        raise ValueError(f"기존 v2 cache shape 오류: {chk.shape}")
    del chk
    cache = np.lib.format.open_memmap(
        EMBED_PATH, mode="r+", dtype=np.float16, shape=shape
    )
else:
    cache = np.lib.format.open_memmap(
        EMBED_PATH, mode="w+", dtype=np.float16, shape=shape
    )

if DONE_PATH.exists():
    done = np.load(DONE_PATH)
    if done.shape != (len(segments),):
        raise ValueError(f"done mask shape 오류: {done.shape}")
else:
    done = np.zeros(len(segments), dtype=bool)

print("Already completed:", int(done.sum()), "/", len(done))
print("Start device:", MERT_DEVICE)

session_start = time.time()
session_start_count = int(done.sum())

for i, row in segments.iterrows():
    if done[i]:
        continue

    y = load_segment_waveform(row)

    try:
        emb = extract_mert_embeddings(y, MERT_DEVICE)
    except RuntimeError as e:
        if MERT_DEVICE.type != "mps":
            raise RuntimeError(f"{row['segment_id']} extraction failed: {e}") from e

        print("\nMPS runtime error -> CPU fallback")
        print(row["segment_id"], str(e)[:500])

        model = model.to("cpu")
        try:
            torch.mps.empty_cache()
        except Exception:
            pass
        gc.collect()

        MERT_DEVICE = torch.device("cpu")
        emb = extract_mert_embeddings(y, MERT_DEVICE)

    if emb.shape != (13, 768) or not np.isfinite(emb).all():
        raise RuntimeError(f"invalid embedding at {row['segment_id']}")

    cache[i] = emb.astype(np.float16)
    done[i] = True

    completed = int(done.sum())

    if completed % 50 == 0 or completed == len(done):
        cache.flush()
        np.save(DONE_PATH, done)

        elapsed = time.time() - session_start
        newly_done = completed - session_start_count
        rate = newly_done / max(elapsed, 1e-9)
        remain = len(done) - completed
        eta_min = remain / max(rate, 1e-9) / 60

        print(
            f"{completed}/{len(done)} | device={MERT_DEVICE} | "
            f"{rate:.2f} seg/s | ETA≈{eta_min:.1f} min"
        )

        if MERT_DEVICE.type == "mps":
            try:
                torch.mps.empty_cache()
            except Exception:
                pass

cache.flush()
np.save(DONE_PATH, done)

index_df = segments[
    [
        "segment_id",
        "track_sample_id",
        "original_audio",
        "label",
        "label_id",
        "genre",
        "generator",
        "split",
    ]
].copy()
index_df["cache_index"] = np.arange(len(index_df))
index_df.to_csv(INDEX_PATH, index=False, encoding="utf-8-sig")

print("Completed :", int(done.sum()))
print("Incomplete:", int((~done).sum()))
print("Cache     :", cache.shape)
print("dtype     :", cache.dtype)
print("End device:", MERT_DEVICE)

# ID와 행 수가 틀리면 특징·라벨 대응이 어긋나므로 여기서 확인한다.
assert bool(done.all())
assert cache.shape == (10077, 13, 768)
print("MERT Embedding Cache QC PASS: True")

Already completed: 0 / 10077
Start device: mps
50/10077 | device=mps | 8.69 seg/s | ETA≈19.2 min
100/10077 | device=mps | 8.66 seg/s | ETA≈19.2 min
150/10077 | device=mps | 8.66 seg/s | ETA≈19.1 min
200/10077 | device=mps | 8.64 seg/s | ETA≈19.0 min
250/10077 | device=mps | 8.66 seg/s | ETA≈18.9 min
300/10077 | device=mps | 8.65 seg/s | ETA≈18.8 min
350/10077 | device=mps | 8.66 seg/s | ETA≈18.7 min
400/10077 | device=mps | 8.65 seg/s | ETA≈18.6 min
450/10077 | device=mps | 8.64 seg/s | ETA≈18.6 min
500/10077 | device=mps | 8.63 seg/s | ETA≈18.5 min
550/10077 | device=mps | 8.63 seg/s | ETA≈18.4 min
600/10077 | device=mps | 8.62 seg/s | ETA≈18.3 min
650/10077 | device=mps | 8.62 seg/s | ETA≈18.2 min
700/10077 | device=mps | 8.62 seg/s | ETA≈18.1 min
750/10077 | device=mps | 8.62 seg/s | ETA≈18.0 min
800/10077 | device=mps | 8.62 seg/s | ETA≈17.9 min
850/10077 | device=mps | 8.62 seg/s | ETA≈17.8 min
900/10077 | device=mps | 8.62 seg/s | ETA≈17.7 min
950/10077 | device=mps | 8.62 seg/s 

## 6. 평가 함수

In [7]:
# ROC에서 REAL 오탐과 FAKE 미탐의 균형을 확인한다.
def find_eer_threshold(y_true, scores):
    fpr, tpr, thresholds = roc_curve(y_true, scores, pos_label=1)
    fnr = 1.0 - tpr
    valid = np.isfinite(thresholds)
    fpr, fnr, thresholds = fpr[valid], fnr[valid], thresholds[valid]
    idx = np.argmin(np.abs(fpr - fnr))
    return {
        "eer": float((fpr[idx] + fnr[idx]) / 2.0),
        "threshold": float(thresholds[idx]),
    }


def evaluate_scores(y_true, scores, threshold):
    y_true = np.asarray(y_true, dtype=int)
    scores = np.asarray(scores, dtype=float)
    pred = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    # EER은 두 오류율이 같아지는 지점이다. 분류에는 Validation에서 정한 임계값을 쓴다.
    eer = find_eer_threshold(y_true, scores)["eer"]

    return {
        "roc_auc": float(roc_auc_score(y_true, scores)),
        "pr_auc": float(average_precision_score(y_true, scores)),
        "eer": float(eer),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, pred)),
        "macro_f1": float(f1_score(y_true, pred, average="macro", zero_division=0)),
        "real_fpr": float(fp / (fp + tn)),
        "fake_miss_rate": float(fn / (fn + tp)),
        "threshold_used": float(threshold),
    }


def make_track_scores(meta, scores):
    temp = meta[
        [
            "track_sample_id",
            "original_audio",
            "label",
            "label_id",
            "genre",
            "generator",
            "split",
        ]
    ].copy()
    temp["score"] = scores

    return temp.groupby("track_sample_id", as_index=False).agg(
        original_audio=("original_audio", "first"),
        label=("label", "first"),
        label_id=("label_id", "first"),
        genre=("genre", "first"),
        generator=("generator", "first"),
        split=("split", "first"),
        segment_count=("score", "size"),
        score=("score", "mean"),
    )

## 7. 13개 layer 탐색 — Validation Track EER 기준

In [8]:
# 배열 캐시를 읽어 저장된 특징과 ID의 순서를 확인한다.
cache = np.load(EMBED_PATH, mmap_mode="r")

# 같은 원곡의 표본이 여러 분할에 섞이지 않도록 저장된 split을 그대로 사용한다.
train_idx = np.flatnonzero(segments["split"].to_numpy() == "train")
val_idx = np.flatnonzero(segments["split"].to_numpy() == "val")
test_idx = np.flatnonzero(segments["split"].to_numpy() == "test")

# ID와 행 수가 틀리면 특징·라벨 대응이 어긋나므로 여기서 확인한다.
assert (len(train_idx), len(val_idx), len(test_idx)) == (6967, 1538, 1572)

y_train = segments.iloc[train_idx]["label_id"].to_numpy()
y_val = segments.iloc[val_idx]["label_id"].to_numpy()
val_meta = segments.iloc[val_idx].reset_index(drop=True)

layer_rows = []
artifacts = {}

for layer in range(13):
    X_train = np.asarray(cache[train_idx, layer, :], dtype=np.float32)
    X_val = np.asarray(cache[val_idx, layer, :], dtype=np.float32)

    # 특징마다 범위가 달라 표준화한다. 평균·표준편차는 Train에서만 구한다.
    scaler = StandardScaler()
    X_train_z = scaler.fit_transform(X_train)
    # Validation/Test에는 Train의 표준화 기준만 적용해 정보 누수를 막는다.
    X_val_z = scaler.transform(X_val)

    clf = LogisticRegression(
        # 클래스별 표본 수가 달라 손실에 기여하는 비중을 조정한다.
        class_weight="balanced",
        max_iter=3000,
        random_state=RANDOM_STATE,
        solver="lbfgs",
    )
    clf.fit(X_train_z, y_train)

    # 확률 출력에서 모델이 예측한 클래스 순서에 맞는 점수 열을 사용한다.
    val_scores = clf.predict_proba(X_val_z)[:, 1]
    # EER은 두 오류율이 같아지는 지점이다. 분류에는 Validation에서 정한 임계값을 쓴다.
    seg_eer = find_eer_threshold(y_val, val_scores)

    val_track = make_track_scores(val_meta, val_scores)
    track_eer = find_eer_threshold(val_track["label_id"], val_track["score"])

    row = {
        "layer": layer,
        "val_segment_roc_auc": roc_auc_score(y_val, val_scores),
        "val_segment_eer": seg_eer["eer"],
        "val_track_roc_auc": roc_auc_score(val_track["label_id"], val_track["score"]),
        "val_track_eer": track_eer["eer"],
        "val_track_threshold": track_eer["threshold"],
    }
    layer_rows.append(row)
    artifacts[layer] = {"scaler": scaler, "clf": clf}

    print(
        f"layer {layer:02d} | "
        f"Val Track AUC={row['val_track_roc_auc']:.4f} | "
        f"EER={row['val_track_eer']:.4f}"
    )

layer_search = pd.DataFrame(layer_rows)
layer_ranked = layer_search.sort_values(
    ["val_track_eer", "val_track_roc_auc"],
    ascending=[True, False],
).reset_index(drop=True)

display(layer_ranked.round(4))

BEST_LAYER = int(layer_ranked.iloc[0]["layer"])
print("Best MERT layer:", BEST_LAYER)

layer 00 | Val Track AUC=0.9331 | EER=0.1169
layer 01 | Val Track AUC=0.9433 | EER=0.1489
layer 02 | Val Track AUC=0.9466 | EER=0.0910
layer 03 | Val Track AUC=0.9468 | EER=0.1365
layer 04 | Val Track AUC=0.9504 | EER=0.1014
layer 05 | Val Track AUC=0.9526 | EER=0.1138
layer 06 | Val Track AUC=0.9710 | EER=0.0962
layer 07 | Val Track AUC=0.9695 | EER=0.1210
layer 08 | Val Track AUC=0.9520 | EER=0.1220
layer 09 | Val Track AUC=0.9581 | EER=0.0900
layer 10 | Val Track AUC=0.9634 | EER=0.1034
layer 11 | Val Track AUC=0.9557 | EER=0.1148
layer 12 | Val Track AUC=0.9472 | EER=0.1313


,layer,val_segment_roc_auc,val_segment_eer,val_track_roc_auc,val_track_eer,val_track_threshold
0,9,0.9544,0.0913,0.9581,0.0900,0.8365
1,2,0.9336,0.1543,0.9466,0.0910,0.7483
2,6,0.9582,0.0847,0.9710,0.0962,0.8874
3,4,0.9398,0.1042,0.9504,0.1014,0.8349
4,10,0.9585,0.1214,0.9634,0.1034,0.8959
5,5,0.9391,0.1400,0.9526,0.1138,0.8882
6,11,0.9499,0.1134,0.9557,0.1148,0.8912
7,0,0.9289,0.1350,0.9331,0.1169,0.7360
8,7,0.9621,0.1204,0.9695,0.1210,0.9608
9,8,0.9504,0.1280,0.9520,0.1220,0.9066


Best MERT layer: 9


## 8. Best layer — Val/Test 최종 평가

In [9]:
best_scaler = artifacts[BEST_LAYER]["scaler"]
best_clf = artifacts[BEST_LAYER]["clf"]

X_val = np.asarray(cache[val_idx, BEST_LAYER, :], dtype=np.float32)
X_test = np.asarray(cache[test_idx, BEST_LAYER, :], dtype=np.float32)

# Validation/Test에는 Train의 표준화 기준만 적용해 정보 누수를 막는다.
val_scores = best_clf.predict_proba(best_scaler.transform(X_val))[:, 1]
# `predict_proba`로 각 클래스의 예측 확률을 얻는다.
# Validation/Test에는 Train의 표준화 기준만 적용해 정보 누수를 막는다.
test_scores = best_clf.predict_proba(best_scaler.transform(X_test))[:, 1]

val_meta = segments.iloc[val_idx].reset_index(drop=True)
test_meta = segments.iloc[test_idx].reset_index(drop=True)

# ROC에서 REAL 오탐과 FAKE 미탐의 균형을 확인한다.
# EER은 두 오류율이 같아지는 지점이다. 분류에는 Validation에서 정한 임계값을 쓴다.
SEGMENT_THRESHOLD = find_eer_threshold(val_meta["label_id"], val_scores)["threshold"]

val_track = make_track_scores(val_meta, val_scores)
test_track = make_track_scores(test_meta, test_scores)

TRACK_THRESHOLD = find_eer_threshold(val_track["label_id"], val_track["score"])[
    "threshold"
]

rows = [
    {
        "model": "MERT95M+LR",
        "level": "segment",
        "split": "val",
        **evaluate_scores(val_meta["label_id"], val_scores, SEGMENT_THRESHOLD),
    },
    {
        "model": "MERT95M+LR",
        "level": "segment",
        "split": "test",
        **evaluate_scores(test_meta["label_id"], test_scores, SEGMENT_THRESHOLD),
    },
    {
        "model": "MERT95M+LR",
        "level": "track",
        "split": "val",
        **evaluate_scores(val_track["label_id"], val_track["score"], TRACK_THRESHOLD),
    },
    {
        "model": "MERT95M+LR",
        "level": "track",
        "split": "test",
        **evaluate_scores(test_track["label_id"], test_track["score"], TRACK_THRESHOLD),
    },
]

mert_metrics = pd.DataFrame(rows)

display(
    mert_metrics[
        [
            "model",
            "level",
            "split",
            "roc_auc",
            "pr_auc",
            "eer",
            "balanced_accuracy",
            "macro_f1",
            "real_fpr",
            "fake_miss_rate",
            "threshold_used",
        ]
    ].round(4)
)

print("Best layer       :", BEST_LAYER)
print("Segment threshold:", SEGMENT_THRESHOLD)
print("Track threshold  :", TRACK_THRESHOLD)

,model,level,split,roc_auc,pr_auc,eer,balanced_accuracy,macro_f1,real_fpr,fake_miss_rate,threshold_used
0,MERT95M+LR,segment,val,0.9544,0.9943,0.0913,0.9087,0.7888,0.0909,0.0917,0.9830
1,MERT95M+LR,segment,test,0.9803,0.9980,0.0729,0.9149,0.7871,0.0741,0.0960,0.9830
2,MERT95M+LR,track,val,0.9581,0.9923,0.0900,0.9100,0.7896,0.0909,0.0890,0.8365
3,MERT95M+LR,track,test,0.9872,0.9987,0.0455,0.9363,0.8107,0.0444,0.0830,0.8365


Best layer       : 9
Segment threshold: 0.9829857246896891
Track threshold  : 0.8364592311843411


## 9. RBF-SVM vs CNN vs MERT — Track Test

In [10]:
baseline = pd.read_csv(PROJECT_ROOT / "results/baseline/baseline_metrics.csv")
cnn = pd.read_csv(PROJECT_ROOT / "results/cnn/cnn_metrics.csv")

svm_test = baseline[
    (baseline["model"] == "RBF-SVM")
    & (baseline["level"] == "track")
    # 같은 원곡의 표본이 여러 분할에 섞이지 않도록 저장된 split을 그대로 사용한다.
    & (baseline["split"] == "test")
].copy()

cnn_test = cnn[
    (cnn["model"] == "LogMelCNN") & (cnn["level"] == "track") & (cnn["split"] == "test")
].copy()

mert_test = mert_metrics[
    (mert_metrics["level"] == "track") & (mert_metrics["split"] == "test")
].copy()

comparison = pd.concat(
    [svm_test, cnn_test, mert_test],
    ignore_index=True,
)

display(
    comparison[
        [
            "model",
            "roc_auc",
            "eer",
            "balanced_accuracy",
            "macro_f1",
            "real_fpr",
            "fake_miss_rate",
        ]
    ].round(4)
)

,model,roc_auc,eer,balanced_accuracy,macro_f1,real_fpr,fake_miss_rate
0,RBF-SVM,0.9644,0.1395,0.8696,0.7502,0.1556,0.1053
1,LogMelCNN,0.9772,0.1042,0.9111,0.8535,0.1333,0.0445
2,MERT95M+LR,0.9872,0.0455,0.9363,0.8107,0.0444,0.0830


## 10. 저장 / 최종 QC

In [11]:
layer_search.to_csv(
    RESULT_DIR / "mert_layer_search_v2.csv",
    index=False,
    encoding="utf-8-sig",
)
mert_metrics.to_csv(
    RESULT_DIR / "mert_metrics_v2.csv",
    index=False,
    encoding="utf-8-sig",
)
comparison.to_csv(
    RESULT_DIR / "mert_vs_svm_cnn_track_test_v2.csv",
    index=False,
    encoding="utf-8-sig",
)

artifact_path = CHECKPOINT_DIR / "mert95m_frozen_lr_v2.joblib"
joblib.dump(
    {
        "model_name": MODEL_NAME,
        "revision": MERT_REVISION,
        "transformers_version": transformers.__version__,
        "best_layer": BEST_LAYER,
        "scaler": best_scaler,
        "classifier": best_clf,
        "segment_threshold": float(SEGMENT_THRESHOLD),
        "track_threshold": float(TRACK_THRESHOLD),
    },
    artifact_path,
)

test_track_row = mert_metrics[
    # 같은 원곡의 표본이 여러 분할에 섞이지 않도록 저장된 split을 그대로 사용한다.
    (mert_metrics["level"] == "track") & (mert_metrics["split"] == "test")
].iloc[0]

qc = pd.DataFrame(
    {
        "check": [
            "segment_rows",
            "cache_complete",
            "cache_shape",
            "layers_evaluated",
            "best_layer_valid",
            "test_track_rows",
            "finite_test_auc",
            "artifact_exists",
            "comparison_models",
        ],
        "value": [
            len(segments),
            bool(done.all()),
            str(cache.shape),
            len(layer_search),
            0 <= BEST_LAYER <= 12,
            len(test_track),
            np.isfinite(test_track_row["roc_auc"]),
            artifact_path.exists(),
            comparison["model"].nunique(),
        ],
    }
)

display(qc)

core_qc_pass = (
    len(segments) == 10077
    and bool(done.all())
    and cache.shape == (10077, 13, 768)
    and len(layer_search) == 13
    and 0 <= BEST_LAYER <= 12
    and len(test_track) == 539
    and np.isfinite(test_track_row["roc_auc"])
    and artifact_path.exists()
    and comparison["model"].nunique() == 3
)

print("===== FINAL RESULT =====")
print("MERT Frozen Baseline Core QC PASS:", core_qc_pass)

,check,value
0,segment_rows,10077
1,cache_complete,True
2,cache_shape,"(10077, 13, 768)"
3,layers_evaluated,13
4,best_layer_valid,True
5,test_track_rows,539
6,finite_test_auc,True
7,artifact_exists,True
8,comparison_models,3


===== FINAL RESULT =====
MERT Frozen Baseline Core QC PASS: True


## 초기 MERT 결과

- MERT-v1-95M의 13개 layer embedding을 10,077개 segment에서 추출했다.
- Validation Track EER로 최적 layer를 선택하고 Logistic Regression을 학습했다.
- Test Track ROC-AUC는 **0.9872**, EER은 **0.0455**, Balanced Accuracy는 **0.9363**이다.
- 기존 RBF-SVM(0.9644)과 Log-Mel CNN(0.9772)보다 높은 in-domain Track ROC-AUC를 기록했다.
- 결과는 `results/mert/`, 모델은 `checkpoints/mert/`에 저장했다.